# DataSage — Stage 2: Enrichment GRPO Training

Trains Qwen2.5-3B via **Unsloth** + **TRL GRPO** to perform multi-domain data enrichment actions against the DataSage Enrichment OpenEnv environment deployed on HuggingFace Spaces.

**Environment:** [ricalanis/datasage-enrichment](https://huggingface.co/spaces/ricalanis/datasage-enrichment)  
**Model:** Qwen2.5-3B-Instruct (4-bit via Unsloth)  
**Framework:** TRL GRPOTrainer with environment-in-the-loop rewards

In [ ]:
!pip install -q "openenv-core[core]>=0.2.1"
!pip install -q trl>=0.26 transformers accelerate peft bitsandbytes
!pip install -q vllm
!pip install -q unsloth
!pip install -q wandb

In [ ]:
import json
import os
import re
import torch
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# === Configuration ===
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-enrichment.hf.space"
HF_MODEL_REPO = "ricalanis/datasage-qwen-enrichment"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["HF_TOKEN"] = "your_token_here"
# os.environ["WANDB_API_KEY"] = "your_key_here"

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")

In [ ]:
SYSTEM_PROMPT = """\
You are a data enrichment agent. You enrich enterprise datasets by adding \
derived fields, lookups, and computed columns across multiple domains \
(HR, Sales, Project Management, IT Operations).

Each turn, analyze the schema and respond with a JSON enrichment action:
{"operation": "<op>", "field_name": "<name>", "source": "<source>", \
"logic": "<logic>", "params": {}}

Available operations:
- add_field: Add a new enrichment field from a known source
- lookup: Look up external reference data
- compute_derived: Compute a derived metric from existing columns
- add_category: Add a categorical classification

Available enrichment sources per domain:
- HR: salary_band, tenure_risk, satisfaction_index, industry_benchmark, flight_risk_score
- Sales: deal_size_category, velocity_score, win_probability_model, industry_code, competitive_risk
- PM: schedule_risk_score, resource_utilization, dependency_chain_depth, burndown_rate, delay_probability
- IT Ops: sla_compliance_flag, mttr_band, escalation_path, incident_severity_score, recurring_pattern_flag

Think step by step: examine the current schema, identify what enrichments \
would add the most value, then act."""

TASK_PROMPTS = [
    # HR domain (8)
    "This HR dataset has employee data with MonthlyIncome but no salary band classification. Add a salary_band enrichment.",
    "The HR data has YearsAtCompany but no risk assessment. Add tenure_risk to identify flight risk by tenure.",
    "We need a satisfaction_index that combines multiple satisfaction factors. Enrich the HR data.",
    "Add industry_benchmark salary data so we can compare our compensation to market rates.",
    "Compute a flight_risk_score combining tenure, satisfaction, and overtime factors for each employee.",
    "The HR dataset needs salary bands for compensation analysis. Which enrichment should be added first?",
    "We want to identify employees at risk of leaving. Add the most relevant enrichment fields.",
    "Enrich this HR data with all available sources to maximize coverage. Start with the most impactful.",
    # Sales domain (8)
    "The Sales pipeline has Amount data but no deal categorization. Add deal_size_category.",
    "We need to track deal velocity. Add velocity_score based on DaysInStage benchmarks.",
    "Add a win_probability_model to predict deal outcomes from stage and velocity data.",
    "The sales data needs industry classification codes. Add industry_code from account patterns.",
    "Compute competitive_risk scores to identify deals at risk of being lost to competitors.",
    "The sales pipeline lacks predictive metrics. Add win_probability_model as the first enrichment.",
    "Enrich deals with size categories and velocity scores for pipeline analysis.",
    "The Sales data enrichment coverage is 0%. Start adding the most valuable enrichment fields.",
    # PM domain (8)
    "The PM dataset has CompletionPct but no schedule risk assessment. Add schedule_risk_score.",
    "We need resource utilization metrics. Add resource_utilization from EstimatedHours and ActualHours.",
    "Add dependency_chain_depth to understand task blocking relationships.",
    "Compute burndown_rate to track project completion velocity against plan.",
    "Add delay_probability to predict which tasks are likely to miss their deadlines.",
    "The PM data needs risk scoring. Add schedule_risk_score as the highest-priority enrichment.",
    "Enrich project tasks with utilization and burndown metrics for sprint analysis.",
    "The PM enrichment coverage is very low. Add fields to improve project analytics.",
    # IT Ops domain (8)
    "The IT Ops data has SLA targets but no compliance flags. Add sla_compliance_flag.",
    "We need MTTR classification. Add mttr_band to categorize resolution speed.",
    "Add escalation_path recommendations based on ticket category and priority.",
    "Compute incident_severity_score from priority and customer impact factors.",
    "Add recurring_pattern_flag to identify tickets that are part of recurring issues.",
    "The IT Ops data needs SLA monitoring. Add sla_compliance_flag as the first enrichment.",
    "Enrich tickets with severity scoring and MTTR bands for operations dashboards.",
    "The IT Ops enrichment coverage is 0%. Start adding enrichments for incident analysis.",
]


def make_conversation(user_msg):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]


dataset = Dataset.from_dict({
    "prompt": [make_conversation(p) for p in TASK_PROMPTS]
})

print(f"Dataset size: {len(dataset)} prompts across 4 domains")

## Reward Functions

Three reward signals drive GRPO training:
1. **Environment Reward** (primary): Steps through the OpenEnv enrichment environment
2. **JSON Format Reward**: Encourages well-structured JSON enrichment actions with valid sources
3. **Reasoning Reward**: Encourages analytical thinking before acting

In [ ]:
def parse_enrichment_action(text):
    """Parse model output into an enrichment action dict."""
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "field_name" in data:
                return data
        except json.JSONDecodeError:
            pass
    text_lower = text.lower()
    sources = ["salary_band", "tenure_risk", "satisfaction_index", "industry_benchmark",
               "flight_risk_score", "deal_size_category", "velocity_score",
               "win_probability_model", "industry_code", "competitive_risk",
               "schedule_risk_score", "resource_utilization", "dependency_chain_depth",
               "burndown_rate", "delay_probability", "sla_compliance_flag", "mttr_band",
               "escalation_path", "incident_severity_score", "recurring_pattern_flag"]
    for source in sources:
        if source.replace("_", " ") in text_lower or source in text_lower:
            return {"operation": "add_field", "field_name": source, "source": source,
                    "logic": "", "params": {}}
    return {"operation": "add_field", "field_name": "unknown", "source": "", "logic": "", "params": {}}


def env_reward_fn(completions, **kwargs):
    """Step through the Enrichment environment for each completion. Primary reward."""
    import requests
    rewards = []
    for text in completions:
        try:
            action_dict = parse_enrichment_action(text)
            requests.post(f"{ENV_URL}/reset", json={})
            step_resp = requests.post(f"{ENV_URL}/step", json={"action": action_dict})
            step_data = step_resp.json()
            rewards.append(float(step_data.get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards


def json_format_reward(completions, **kwargs):
    """Reward well-formed JSON enrichment actions."""
    rewards = []
    valid_ops = {"add_field", "lookup", "compute_derived", "add_category"}
    valid_sources = {
        "salary_band", "tenure_risk", "satisfaction_index", "industry_benchmark",
        "flight_risk_score", "deal_size_category", "velocity_score",
        "win_probability_model", "industry_code", "competitive_risk",
        "schedule_risk_score", "resource_utilization", "dependency_chain_depth",
        "burndown_rate", "delay_probability", "sla_compliance_flag", "mttr_band",
        "escalation_path", "incident_severity_score", "recurring_pattern_flag",
    }
    for text in completions:
        if re.search(r'\{[^{}]*"operation"[^{}]*\}', text):
            try:
                match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text)
                data = json.loads(match.group())
                op_ok = data.get("operation") in valid_ops
                field_ok = "field_name" in data and data["field_name"] != "unknown"
                source_ok = data.get("source", "") in valid_sources
                if op_ok and field_ok and source_ok:
                    rewards.append(1.0)
                elif op_ok and field_ok:
                    rewards.append(0.6)
                elif op_ok:
                    rewards.append(0.3)
                else:
                    rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


def reasoning_reward(completions, **kwargs):
    """Reward chain-of-thought reasoning before the enrichment action."""
    rewards = []
    for text in completions:
        score = 0.0
        lower = text.lower()
        if any(w in lower for w in ["first", "let me", "i should", "step 1", "think", "analyze", "examining"]):
            score += 0.3
        if any(w in lower for w in ["enrich", "add", "derive", "compute", "coverage", "value", "analytical"]):
            score += 0.2
        rewards.append(min(score, 0.5))
    return rewards

## GRPO Training

Using Group Relative Policy Optimization with environment-in-the-loop rewards. Config tuned for Colab T4 GPU (use larger batches on H100).

In [ ]:
training_args = GRPOConfig(
    output_dir="./outputs/enrichment-grpo",
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=512,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb",
    run_name="datasage-enrichment-grpo",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        env_reward_fn,
        json_format_reward,
        reasoning_reward,
    ],
)

print("Starting Stage 2 (Enrichment) GRPO training...")
trainer.train()

In [ ]:
output_dir = "./outputs/enrichment-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Training complete! Model saved to {output_dir}")

trainer.push_to_hub(HF_MODEL_REPO)
print(f"Model pushed to https://huggingface.co/{HF_MODEL_REPO}")